<a href="https://colab.research.google.com/github/MuizSarwar/Deep-Learning-study-materials-/blob/main/CNN_project_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
import kagglehub

#Download Dataset

In [2]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


#Setup

In [3]:
torch.manual_seed(42)
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(f" Using Device {device}")

 Using Device cuda


In [4]:
TRAIN_PATH = os.path.join(path, "PlantVillage","train")
VAL_PATH = os.path.join(path, "PlantVillage","val")

#Transformation

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3,std=[0.5]*3)
    ]
)

#Custom Data

In [6]:
class MultiClassClassification(Dataset):
  def __init__(self,root_dir,transform=None):
    super().__init__()

    self.root_dir = root_dir
    self.samples = []
    self.transform = transform
    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
    self.class_and_index = {class_name:idx for idx,class_name in enumerate(self.classes)}

    #defined for validation:
    valid_ext = (".jpg", ".jpeg", ".png")


    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)
      #iterate over each img in the class
      for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path,image_name)

        if os.path.isfile(image_path) and image_name.lower().endswith(valid_ext):
          label = self.class_and_index[class_name]
          self.samples.append((image_path,label))


  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    image_path,label = self.samples[index]
    image = Image.open(image_path).convert("RGB")
    if self.transform:
      image = self.transform(image)

    return image,label

#Load Data

In [7]:
#load full data
train_dataset_full = MultiClassClassification(TRAIN_PATH,transform)
test_dataset_full = MultiClassClassification(VAL_PATH,transform)

In [8]:
#See total class in the dataset:
total_classes = len(train_dataset_full.classes)
print(total_classes)

38


In [9]:
#subset(small part) of full data
train_dataset = Subset(train_dataset_full, list(range(min(100, len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(100, len(test_dataset_full)))))

#DataLoader

In [10]:
pin = True if device.type == 'cuda' else False

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

##CNN Architecture

In [11]:
class MyCNN(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.features = nn.Sequential(
        #layer-1
        nn.Conv2d(3,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),

        #layer-2
        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),

        #layer-3
        nn.Conv2d(64,128,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),

        #Layer-1
        nn.Linear(128*16*16,128),
        nn.ReLU(),
        nn.Dropout(0.4),

        #Layer-2
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(0.4),

        #Layer-3
        nn.Linear(64,num_classes)

    )

  def forward(self,x):
      extracted_features = self.features(x)
      output = self.classifier(extracted_features)
      return output

#Create the model

In [12]:
model = MyCNN(num_classes=total_classes).to(device)

#Training Setup

In [13]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

#Training Loop

In [14]:
for epoch in range(epochs):
  model.train()   #what is the pourpose of this line ?
  total_loss = 0

  for batch_features,batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)
    loss = criterion(outputs,batch_labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += loss.item()

  avg_loss = total_loss/len(train_loader)
  print(f"Loss of epoch-{epoch+1} is {avg_loss:.5f}")


Loss of epoch-1 is 1.00701
Loss of epoch-2 is 0.01433
Loss of epoch-3 is 0.00002
Loss of epoch-4 is 0.00000
Loss of epoch-5 is 0.00001
Loss of epoch-6 is 0.00000
Loss of epoch-7 is 0.00000
Loss of epoch-8 is 0.00000
Loss of epoch-9 is 0.00000
Loss of epoch-10 is 0.00000


#Evaluation

In [15]:
def evaluate(loader):
  model.eval()
  total,correct = 0,0

  with torch.no_grad():
     for batch_features, batch_labels in loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
  return correct / total


In [16]:
test_accuracy = evaluate(test_loader)
print(test_accuracy)

1.0
